# Tiền xử lý dữ liệu chuyến bay và thời tiết (NB, DN, TSN)
Notebook này thực hiện đầy đủ các bước tiền xử lý cho bài toán dự đoán delay:
- Nạp dữ liệu raw từ 3 sân bay và weather
- Chuẩn hóa schema, thời gian, trạng thái
- Tạo nhãn delay và gom snapshot
- Ghép weather theo thời gian gần nhất trước thời điểm crawl
- Tạo feature và xuất dữ liệu processed

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

RANDOM_STATE = 42
DELAY_THRESHOLD_MINUTES = 15
TRAINING_SNAPSHOT_LEAD_MINUTES = 30

## Bước 1 - Nạp dữ liệu raw từ PostgreSQL
Bước này tự động đọc 4 bảng từ PostgreSQL: flights_nb, flights_dn, flights_tsn và weather_metar.
Mục tiêu là tạo bảng flights_raw gồm dữ liệu hợp nhất của 3 sân bay.

In [2]:
import os
import sys
from pathlib import Path


def find_project_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "Data Collection + Processing",
        Path.cwd().parent / "Data Collection + Processing",
        Path.cwd().parent.parent / "Data Collection + Processing",
    ]
    for candidate in candidates:
        if (candidate / "db_utils.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return candidate
    raise FileNotFoundError("Khong tim thay db_utils.py")


def load_env_file(env_path: Path) -> None:
    """Load KEY=VALUE pairs into os.environ (minimal .env loader, no deps)."""
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        existing = os.environ.get(key)
        if existing is None or existing.strip() == "":
            os.environ[key] = value


project_dir = find_project_dir()
load_env_file(project_dir / ".env")

# Debug (khong in ra secret): chi bao co/khong
print("DATABASE_URL loaded:", "yes" if os.getenv("DATABASE_URL") else "no")

from db_utils import load_table, save_dataframe

processed_dir = project_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

flight_tables = {
    "NB": "flights_nb",
    "DN": "flights_dn",
    "TSN": "flights_tsn",
}

flight_frames = []
for airport_code, table_name in flight_tables.items():
    df_flights = load_table(table_name)
    if df_flights.empty:
        print(f"Canh bao: bang {table_name} khong co du lieu.")
        continue
    df_flights = df_flights.drop(columns=["created_at"], errors="ignore")
    df_flights["source_airport"] = airport_code
    flight_frames.append(df_flights)

if not flight_frames:
    raise RuntimeError("Khong tim thay du lieu flight trong PostgreSQL.")

flights_raw = pd.concat(flight_frames, ignore_index=True)
weather_raw = load_table("weather_metar").drop(columns=["created_at"], errors="ignore")

print("Flights raw shape:", flights_raw.shape)
print("Weather raw shape:", weather_raw.shape)
flights_raw.head(3)

DATABASE_URL loaded: yes


/mnt/c/Users/ADMIN/Desktop/IDA/data-monitoring-lab/Data Collection + Processing/db_utils.py:53: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query.as_string(conn), conn)
/mnt/c/Users/ADMIN/Desktop/IDA/data-monitoring-lab/Data Collection + Processing/db_utils.py:53: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query.as_string(conn), conn)
/mnt/c/Users/ADMIN/Desktop/IDA/data-monitoring-lab/Data Collection + Processing/db_utils.py:53: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchem

Flights raw shape: (31943, 9)
Weather raw shape: (3978, 10)


,data_retrieved_at_vn,flight_date,direction,scheduled_time,estimated_time,airport,flight_number,status,source_airport
0,2026-04-17 22:01:38,2026-04-17,Arrival,00:10,00:23,HO CHI MINH,VN224,Arrived,NB
1,2026-04-17 22:01:38,2026-04-17,Arrival,00:10,23:58-,HO CHI MINH,9G886,,NB
2,2026-04-17 22:01:38,2026-04-17,Arrival,00:25,00:18,HO CHI MINH,VJ1128,Arrived,NB


## Bước 2 - Chuẩn hóa cột và giá trị text
Bước này chuẩn hóa tên cột về dạng snake_case, làm sạch chuỗi, đổi tên các cột theo schema thống nhất, và map trạng thái chuyến bay về nhóm để tạo nhãn.

In [3]:
def normalize_space(x):
    if pd.isna(x):
        return np.nan
    return re.sub(r"\s+", " ", str(x)).strip()


def normalize_flight_number(x):
    if pd.isna(x):
        return np.nan
    s = normalize_space(x).upper().replace(" ", "")
    s = s.replace("-", "")
    return s


def normalize_route_airport(x):
    if pd.isna(x):
        return np.nan
    s = normalize_space(x).upper()
    accent_map = str.maketrans({
        "À": "A", "Á": "A", "Ả": "A", "Ã": "A", "Ạ": "A",
        "Ă": "A", "Ằ": "A", "Ắ": "A", "Ẳ": "A", "Ẵ": "A", "Ặ": "A",
        "Â": "A", "Ầ": "A", "Ấ": "A", "Ẩ": "A", "Ẫ": "A", "Ậ": "A",
        "Đ": "D",
        "È": "E", "É": "E", "Ẻ": "E", "Ẽ": "E", "Ẹ": "E",
        "Ê": "E", "Ề": "E", "Ế": "E", "Ể": "E", "Ễ": "E", "Ệ": "E",
        "Ì": "I", "Í": "I", "Ỉ": "I", "Ĩ": "I", "Ị": "I",
        "Ò": "O", "Ó": "O", "Ỏ": "O", "Õ": "O", "Ọ": "O",
        "Ô": "O", "Ồ": "O", "Ố": "O", "Ổ": "O", "Ỗ": "O", "Ộ": "O",
        "Ơ": "O", "Ờ": "O", "Ớ": "O", "Ở": "O", "Ỡ": "O", "Ợ": "O",
        "Ù": "U", "Ú": "U", "Ủ": "U", "Ũ": "U", "Ụ": "U",
        "Ư": "U", "Ừ": "U", "Ứ": "U", "Ử": "U", "Ữ": "U", "Ự": "U",
        "Ỳ": "Y", "Ý": "Y", "Ỷ": "Y", "Ỹ": "Y", "Ỵ": "Y",
    })
    s = s.translate(accent_map)
    s = re.sub(r"\s+", " ", s).strip()
    replacements = {
        "TP. HO CHI MINH": "HO CHI MINH",
        "HO CHI MINH": "HO CHI MINH",
        "HA NOI": "HA NOI",
        "DA NANG": "DA NANG",
        "BUON MA THUOT": "BUON MA THUOT",
        "CAN THO": "CAN THO",
        "HAI PHONG": "HAI PHONG",
        "THANH HOA": "THANH HOA",
        "PHU QUOC": "PHU QUOC",
        "CON DAO": "CON DAO",
        "QUY NHON": "QUY NHON",
        "HUE": "HUE",
        "NHA TRANG": "NHA TRANG",
        "PLEIKU": "PLEIKU",
        "CHU LAI": "CHU LAI",
        "CAM RANH": "CAM RANH",
        "VINH": "VINH",
    }
    for old_text, new_text in replacements.items():
        s = s.replace(old_text, new_text)
    s = re.sub(r"\s*\(([^)]+)\)", lambda m: f" ({m.group(1).upper()})", s)
    return s


status_map = {
    "da ha canh": "landed",
    "arrived": "landed",
    "landed": "landed",
    "den tre": "delayed",
    "cham": "delayed",
    "delayed": "delayed",
    "dang bay": "enroute",
    "on time": "on_time",
    "dung gio": "on_time",
    "khoi hanh": "departed",
    "departed": "departed",
    "huy": "cancelled",
    "cancelled": "cancelled",
}


def normalize_status(status_text):
    s = normalize_space(status_text)
    if pd.isna(s) or s == "":
        return "unknown"
    s_ascii = (
        s.lower()
        .replace("đ", "d")
        .replace("á", "a").replace("à", "a").replace("ả", "a").replace("ã", "a").replace("ạ", "a")
        .replace("ă", "a").replace("ắ", "a").replace("ằ", "a").replace("ẳ", "a").replace("ẵ", "a").replace("ặ", "a")
        .replace("â", "a").replace("ấ", "a").replace("ầ", "a").replace("ẩ", "a").replace("ẫ", "a").replace("ậ", "a")
        .replace("é", "e").replace("è", "e").replace("ẻ", "e").replace("ẽ", "e").replace("ẹ", "e")
        .replace("ê", "e").replace("ế", "e").replace("ề", "e").replace("ể", "e").replace("ễ", "e").replace("ệ", "e")
        .replace("í", "i").replace("ì", "i").replace("ỉ", "i").replace("ĩ", "i").replace("ị", "i")
        .replace("ó", "o").replace("ò", "o").replace("ỏ", "o").replace("õ", "o").replace("ọ", "o")
        .replace("ô", "o").replace("ố", "o").replace("ồ", "o").replace("ổ", "o").replace("ỗ", "o").replace("ộ", "o")
        .replace("ơ", "o").replace("ớ", "o").replace("ờ", "o").replace("ở", "o").replace("ỡ", "o").replace("ợ", "o")
        .replace("ú", "u").replace("ù", "u").replace("ủ", "u").replace("ũ", "u").replace("ụ", "u")
        .replace("ư", "u").replace("ứ", "u").replace("ừ", "u").replace("ử", "u").replace("ữ", "u").replace("ự", "u")
        .replace("ý", "y").replace("ỳ", "y").replace("ỷ", "y").replace("ỹ", "y").replace("ỵ", "y")
    )

    for k, v in status_map.items():
        if k in s_ascii:
            return v
    return "other"


col_rename_csv = {
    "Data Retrieved At (VN)": "retrieved_at_vn",
    "Flight Date": "flight_date",
    "Direction": "direction",
    "Scheduled Time": "scheduled_time",
    "Estimated Time": "estimated_time",
    "Airport": "route_airport",
    "Flight Number": "flight_number",
    "Status": "status_raw",
    "Source Airport": "source_airport",
}


col_rename_db = {
    "data_retrieved_at_vn": "retrieved_at_vn",
    "airport": "route_airport",
    "status": "status_raw",
}


# Schema-aware rename: CSV headers vs Postgres snake_case
if set(col_rename_csv).intersection(flights_raw.columns):
    col_rename = col_rename_csv
else:
    col_rename = col_rename_db


flights = flights_raw.rename(columns=col_rename).copy()


required_cols = [
    "retrieved_at_vn",
    "flight_date",
    "direction",
    "scheduled_time",
    "estimated_time",
    "route_airport",
    "flight_number",
    "status_raw",
    "source_airport",
]
missing_cols = [c for c in required_cols if c not in flights.columns]
if missing_cols:
    raise KeyError(
        f"Missing columns after renaming: {missing_cols}. "
        f"Available columns: {list(flights.columns)}"
    )


for c in ["source_airport", "direction", "route_airport", "status_raw", "flight_number", "scheduled_time", "estimated_time"]:
    flights[c] = flights[c].apply(normalize_space)


flights["flight_number"] = flights["flight_number"].apply(normalize_flight_number)
flights["direction"] = flights["direction"].str.title()
flights["route_airport_std"] = flights["route_airport"].apply(normalize_route_airport)
flights["status_group"] = flights["status_raw"].apply(normalize_status)


other_statuses = (
    flights.loc[flights["status_group"].eq("other"), "status_raw"]
    .dropna()
    .astype(str)
    .str.strip()
 )
if not other_statuses.empty:
    print("Top status_raw mapped to 'other':")
    print(other_statuses.value_counts().head(20))


flights[["source_airport", "direction", "route_airport", "route_airport_std", "flight_number", "status_raw", "status_group"]].head(5)

Top status_raw mapped to 'other':
status_raw
Scheduled                      7318
Đã đến                         1746
3-16                            802
27-34                           683
Mời hành khách làm thủ tục      239
35-39                           151
43-45                            81
21-24                            64
17-20                            60
Chưa xác định                    55
Hành khách cuối làm thủ tục      47
Hành khách lên tàu bay           46
Hành khách cuối lên tàu bay      43
Estimated dep 15:30              38
Estimated dep 08:00              36
Estimated dep 05:45              34
Estimated dep 07:05              32
Estimated dep 06:00              30
Estimated dep 14:00              29
Estimated dep 20:30              26
Name: count, dtype: int64


,source_airport,direction,route_airport,route_airport_std,flight_number,status_raw,status_group
0,NB,Arrival,HO CHI MINH,HO CHI MINH,VN224,Arrived,landed
1,NB,Arrival,HO CHI MINH,HO CHI MINH,9G886,,unknown
2,NB,Arrival,HO CHI MINH,HO CHI MINH,VJ1128,Arrived,landed
3,NB,Arrival,CAM RANH,CAM RANH,VJ7614,Arrived,landed
4,NB,Arrival,HO CHI MINH,HO CHI MINH,9G894,Arrived,landed


## Bước 3 - Chuẩn hóa datetime và tạo delay_minutes
Bước này parse cột ngày giờ thành datetime, xử lý trường hợp qua ngày (giờ dự kiến < giờ lịch), sau đó tính delay_minutes và tạo nhãn label_delay.

In [4]:
def parse_hhmm(s):
    s = normalize_space(s)
    if pd.isna(s) or s == "":
        return pd.NaT
    dt = pd.to_datetime(s, format="%H:%M", errors="coerce")
    return dt


flights["retrieved_at_vn"] = pd.to_datetime(flights["retrieved_at_vn"], errors="coerce")
flights["flight_date"] = pd.to_datetime(flights["flight_date"], errors="coerce").dt.date

sched_parsed = flights["scheduled_time"].apply(parse_hhmm)
est_parsed = flights["estimated_time"].apply(parse_hhmm)

flights["scheduled_dt"] = pd.to_datetime(flights["flight_date"].astype(str) + " " + sched_parsed.dt.strftime("%H:%M"), errors="coerce")
flights["estimated_dt"] = pd.to_datetime(flights["flight_date"].astype(str) + " " + est_parsed.dt.strftime("%H:%M"), errors="coerce")

# Chi coi la qua ngay neu chuyến khuya va gio du kien nam sang som hom sau.
sched_hour = flights["scheduled_dt"].dt.hour
est_hour = flights["estimated_dt"].dt.hour
cross_day_mask = (
    flights["estimated_dt"].notna()
    & flights["scheduled_dt"].notna()
    & (flights["estimated_dt"] < flights["scheduled_dt"])
    & (sched_hour >= 18)
    & (est_hour <= 6)
)
flights.loc[cross_day_mask, "estimated_dt"] = flights.loc[cross_day_mask, "estimated_dt"] + pd.Timedelta(days=1)

flights["delay_minutes"] = (flights["estimated_dt"] - flights["scheduled_dt"]).dt.total_seconds() / 60.0

# Cat nguong delay bat thuong de giam anh huong du lieu loi.
flights.loc[flights["delay_minutes"].abs() > 600, "delay_minutes"] = np.nan

flights["label_delay"] = np.where(
    flights["delay_minutes"].notna(),
    (flights["delay_minutes"] >= DELAY_THRESHOLD_MINUTES).astype(int),
    np.nan,
)

status_delay_mask = flights["status_group"].eq("delayed") & flights["label_delay"].isna()
status_non_delay_mask = flights["status_group"].isin(["on_time", "landed", "departed"]) & flights["label_delay"].isna()
flights.loc[status_delay_mask, "label_delay"] = 1
flights.loc[status_non_delay_mask, "label_delay"] = 0

flights[["scheduled_dt", "estimated_dt", "delay_minutes", "status_group", "label_delay"]].head(10)

,scheduled_dt,estimated_dt,delay_minutes,status_group,label_delay
0,2026-04-17 00:10:00,2026-04-17 00:23:00,13.0,landed,0.0
1,2026-04-17 00:10:00,NaT,NaN,unknown,NaN
2,2026-04-17 00:25:00,2026-04-17 00:18:00,-7.0,landed,0.0
3,2026-04-17 00:45:00,2026-04-17 00:48:00,3.0,landed,0.0
4,2026-04-17 00:55:00,2026-04-17 00:40:00,-15.0,landed,0.0
5,2026-04-17 01:10:00,2026-04-17 00:52:00,-18.0,landed,0.0
6,2026-04-17 05:30:00,2026-04-17 05:30:00,0.0,landed,0.0
7,2026-04-17 07:10:00,2026-04-17 07:03:00,-7.0,landed,0.0
8,2026-04-17 07:30:00,2026-04-17 07:11:00,-19.0,landed,0.0
9,2026-04-17 07:55:00,2026-04-17 08:04:00,9.0,landed,0.0


## Bước 4 - Khóa chuyến bay, loại trùng lặp và tạo snapshot

Bước này tạo `flight_key` để gom các lần crawl cùng một chuyến bay, loại bản ghi trùng hoàn toàn, sau đó tạo 2 tập:

- flights_current: snapshot mới nhất của mỗi chuyến (phục vụ dashboard realtime)

- flights_training_snapshot: snapshot ưu tiên trước giờ bay 30 phút (phục vụ train model)



Ngoài ra, để phục vụ train, chỉ giữ các bản ghi có `status_raw` thay đổi so với lần crawl trước ("status mới").

In [5]:
# Bước 4: Gom theo flight_key + tạo snapshot (current & training)
flights = flights.copy()

# --- Basic record-level cleaning (thiết yếu cho snapshot/training) ---
n0 = len(flights)
flights = flights.drop_duplicates().copy()
n1 = len(flights)

required_non_null = [
    "source_airport",
    "direction",
    "route_airport_std",
    "flight_number",
    "scheduled_dt",
    "retrieved_at_vn",
    "status_raw",
    "status_group",
]
missing_required_mask = flights[required_non_null].isna().any(axis=1)
n_missing_required = int(missing_required_mask.sum())
flights = flights.loc[~missing_required_mask].copy()

# Direction sanity
valid_directions = {"Arrival", "Departure"}
invalid_dir_mask = ~flights["direction"].isin(valid_directions)
n_invalid_dir = int(invalid_dir_mask.sum())
flights = flights.loc[~invalid_dir_mask].copy()

# Flight number non-empty sanity
flight_number_blank_mask = flights["flight_number"].astype(str).str.strip().eq("")
n_blank_flight_number = int(flight_number_blank_mask.sum())
flights = flights.loc[~flight_number_blank_mask].copy()

print("[QC] flights input:", n0)
print("[QC] dropped duplicate rows:", n0 - n1)
print("[QC] dropped missing required fields:", n_missing_required)
print("[QC] dropped invalid direction:", n_invalid_dir)
print("[QC] dropped blank flight_number:", n_blank_flight_number)
print("[QC] flights after basic cleaning:", len(flights))

# --- Create keys & snapshots ---
key_cols = ["source_airport", "direction", "route_airport_std", "flight_number", "scheduled_dt"]
flights["flight_key"] = flights[key_cols].astype(str).agg("|".join, axis=1)

# Sort theo thời gian crawl
flights = flights.sort_values(["flight_key", "retrieved_at_vn"]).reset_index(drop=True)

# Dedup nhẹ: cùng flight_key + retrieved_at_vn giữ bản ghi cuối
flights_dedup = flights.drop_duplicates(subset=["flight_key", "retrieved_at_vn"], keep="last").copy()

# Snapshot current (realtime): lấy bản ghi mới nhất (không bắt buộc status phải đổi)
flights_current = (
    flights_dedup.sort_values(["flight_key", "retrieved_at_vn"])
    .groupby("flight_key", as_index=False)
    .tail(1)
    .reset_index(drop=True)
 )

# Chỉ lấy những dòng có status mới (và status không rỗng) để đưa vào train
status_now = flights_dedup["status_raw"].fillna("").astype(str).str.strip()
status_prev = (
    flights_dedup.groupby("flight_key")["status_raw"]
    .shift(1)
    .fillna("")
    .astype(str)
    .str.strip()
 )
flights_for_train = flights_dedup[status_now.ne("") & status_now.ne(status_prev)].copy()

lead = pd.Timedelta(minutes=TRAINING_SNAPSHOT_LEAD_MINUTES)

def pick_training_snapshot(g: pd.DataFrame) -> pd.Series:
    g = g.sort_values("retrieved_at_vn")
    scheduled_dt = g["scheduled_dt"].iloc[0]
    target_time = scheduled_dt - lead

    cand = g[g["retrieved_at_vn"].notna() & (g["retrieved_at_vn"] <= target_time)]
    if not cand.empty:
        return cand.iloc[-1]

    # fallback: gần nhất trước giờ bay
    cand2 = g[g["retrieved_at_vn"].notna() & (g["retrieved_at_vn"] <= scheduled_dt)]
    if not cand2.empty:
        return cand2.iloc[-1]

    return g.iloc[0]

flights_training_snapshot = (
    flights_for_train.groupby("flight_key", group_keys=False)
    .apply(pick_training_snapshot)
    .reset_index(drop=True)
 )

print("Flights after dedup:", flights_dedup.shape)
print("Flights current:", flights_current.shape)
print("Flights for train (status updates):", flights_for_train.shape)
print("Flights training snapshot:", flights_training_snapshot.shape)

flights_training_snapshot[["source_airport", "flight_number", "scheduled_dt", "retrieved_at_vn", "status_raw", "delay_minutes", "label_delay"]].head(5)

[QC] flights input: 31943
[QC] dropped duplicate rows: 0
[QC] dropped missing required fields: 10
[QC] dropped invalid direction: 0
[QC] dropped blank flight_number: 0
[QC] flights after basic cleaning: 31933
Flights after dedup: (31792, 16)
Flights current: (20155, 16)
Flights for train (status updates): (24659, 16)
Flights training snapshot: (19735, 16)


,source_airport,flight_number,scheduled_dt,retrieved_at_vn,status_raw,delay_minutes,label_delay
0,DN,VN1910,2026-05-12 08:40:00,2026-05-12 06:28:36,Scheduled,NaN,NaN
1,DN,VN1910,2026-04-18 08:40:00,2026-04-18 00:10:37,Đúng giờ,NaN,0.0
2,DN,VN1910,2026-04-21 08:40:00,2026-04-21 00:50:03,Đúng giờ,NaN,0.0
3,DN,VN1910,2026-04-23 08:40:00,2026-04-23 00:30:21,Đúng giờ,NaN,0.0
4,DN,VN1910,2026-04-25 08:40:00,2026-04-25 00:42:51,Đúng giờ,NaN,0.0


## Bước 5 - Tiền xử lý weather
Bước này chuẩn hóa weather: đổi tên cột, parse giờ báo cáo, map ICAO thành mã sân bay NB/DN/TSN, và chuẩn hóa visibility, gió.

In [6]:
weather = weather_raw.rename(
    columns={
        "icao_code": "icao_code",
        "report_time_utc": "report_time_utc",
        "report_time_vn": "report_time_vn",
        "temperature_c": "temperature_c",
        "dew_point_c": "dew_point_c",
        "wind_direction_deg": "wind_direction_deg",
        "wind_speed_kt": "wind_speed_kt",
        "visibility_miles": "visibility_miles",
        "cloud_cover": "cloud_cover",
        "raw_metar": "raw_metar",
    },
).copy()

icao_map = {
    "VVNB": "NB",
    "VVDN": "DN",
    "VVTS": "TSN",
}

weather["source_airport"] = weather["icao_code"].map(icao_map)
weather["report_time_vn"] = pd.to_datetime(weather["report_time_vn"], errors="coerce")

# Coerce numeric weather columns (DB đôi khi trả về kiểu text)
for c in ["temperature_c", "dew_point_c", "wind_speed_kt"]:
    if c in weather.columns:
        weather[c] = pd.to_numeric(weather[c], errors="coerce")

weather["visibility_miles"] = (
    weather["visibility_miles"]
    .astype(str)
    .str.replace("+", "", regex=False)
)
weather["visibility_miles"] = pd.to_numeric(weather["visibility_miles"], errors="coerce")

weather["wind_direction_deg"] = weather["wind_direction_deg"].replace({"VRB": np.nan})
weather["wind_direction_deg"] = pd.to_numeric(weather["wind_direction_deg"], errors="coerce")
weather["is_wind_variable"] = weather_raw["wind_direction_deg"].astype(str).eq("VRB").astype(int)

weather = weather.dropna(subset=["source_airport", "report_time_vn"]).sort_values(["source_airport", "report_time_vn"])
weather = weather.drop_duplicates(subset=["source_airport", "report_time_vn", "raw_metar"], keep="last")

weather.head(3)

,icao_code,report_time_utc,report_time_vn,temperature_c,dew_point_c,wind_direction_deg,wind_speed_kt,visibility_miles,cloud_cover,raw_metar,source_airport,is_wind_variable
143,VVDN,2026-04-15T16:00:00.000Z,2026-04-15 23:00:00,27,24,190.0,2,6.0,clear,METAR VVDN 151600Z 19002KT CAVOK 27/24 Q1008 N...,DN,0
138,VVDN,2026-04-15T16:30:00.000Z,2026-04-15 23:30:00,26,22,200.0,3,6.0,clear,METAR VVDN 151630Z 20003KT 170V240 CAVOK 26/22...,DN,0
136,VVDN,2026-04-15T17:00:00.000Z,2026-04-16 00:00:00,26,23,190.0,3,6.0,clear,METAR VVDN 151700Z 19003KT CAVOK 26/23 Q1007 N...,DN,0


## Bước 6 - Ghép weather vào snapshot flight
Bước này dùng merge_asof để ghép bản tin weather gần nhất trước thời điểm retrieved_at_vn cho từng source_airport.

In [7]:
def merge_weather_asof(
    flight_df: pd.DataFrame,
    weather_df: pd.DataFrame,
    tolerance: pd.Timedelta = pd.Timedelta(hours=3),
 ) -> pd.DataFrame:
    left = flight_df.sort_values(["source_airport", "retrieved_at_vn"]).copy()
    right = weather_df.sort_values(["source_airport", "report_time_vn"]).copy()


    merged_parts = []
    wx_cols = [
        "temperature_c",
        "dew_point_c",
        "wind_direction_deg",
        "wind_speed_kt",
        "visibility_miles",
        "cloud_cover",
        "is_wind_variable",
        "raw_metar",
        "report_time_vn",
    ]


    for airport_code, g_left in left.groupby("source_airport"):
        g_right = right[right["source_airport"] == airport_code]
        if g_right.empty:
            g_left = g_left.copy()
            for c in wx_cols:
                g_left[c] = np.nan
            g_left["weather_age_minutes"] = np.nan
            merged_parts.append(g_left)
            continue


        g_merged = pd.merge_asof(
            g_left.sort_values("retrieved_at_vn"),
            g_right.sort_values("report_time_vn"),
            left_on="retrieved_at_vn",
            right_on="report_time_vn",
            direction="backward",
            allow_exact_matches=True,
            tolerance=tolerance,
            suffixes=("", "_wx"),
        )


        if "source_airport_wx" in g_merged.columns:
            g_merged = g_merged.drop(columns=["source_airport_wx"])


        g_merged["weather_age_minutes"] = (
            (g_merged["retrieved_at_vn"] - g_merged["report_time_vn"])
            .dt.total_seconds()
            / 60.0
        )


        merged_parts.append(g_merged)


    return pd.concat(merged_parts, ignore_index=True)


def impute_weather(
    df: pd.DataFrame,
    tolerance: pd.Timedelta,
 ) -> pd.DataFrame:
    out = df.sort_values(["source_airport", "retrieved_at_vn"]).copy()
    wx_cols = [
        "temperature_c",
        "dew_point_c",
        "wind_direction_deg",
        "wind_speed_kt",
        "visibility_miles",
        "cloud_cover",
        "is_wind_variable",
        "raw_metar",
        "report_time_vn",
    ]
    max_age = tolerance.total_seconds() / 60.0


    out[wx_cols] = out.groupby("source_airport")[wx_cols].ffill()
    out["weather_age_minutes"] = (
        (out["retrieved_at_vn"] - out["report_time_vn"]).dt.total_seconds() / 60.0
    )
    stale_mask = out["weather_age_minutes"].gt(max_age)
    out.loc[stale_mask, wx_cols] = np.nan
    out.loc[stale_mask, "weather_age_minutes"] = np.nan


    num_cols = ["temperature_c", "dew_point_c", "wind_speed_kt", "visibility_miles"]
    for c in num_cols:
        if c in out.columns:
            med = out.groupby("source_airport")[c].transform("median")
            out[c] = out[c].fillna(med)


    return out


WX_TOLERANCE = pd.Timedelta(hours=3)


current_with_weather = merge_weather_asof(flights_current, weather, tolerance=WX_TOLERANCE)
train_with_weather = merge_weather_asof(flights_training_snapshot, weather, tolerance=WX_TOLERANCE)


current_with_weather = impute_weather(current_with_weather, tolerance=WX_TOLERANCE)
train_with_weather = impute_weather(train_with_weather, tolerance=WX_TOLERANCE)


print("Current with weather:", current_with_weather.shape)
print("Train with weather:", train_with_weather.shape)
print("Weather match rate (train):", round(train_with_weather["report_time_vn"].notna().mean() * 100, 2), "%")
train_with_weather.head(3)

Current with weather: (20155, 28)
Train with weather: (19735, 28)
Weather match rate (train): 100.0 %


,retrieved_at_vn,flight_date,direction,scheduled_time,estimated_time,route_airport,flight_number,status_raw,source_airport,route_airport_std,status_group,scheduled_dt,estimated_dt,delay_minutes,label_delay,flight_key,icao_code,report_time_utc,report_time_vn,temperature_c,dew_point_c,wind_direction_deg,wind_speed_kt,visibility_miles,cloud_cover,raw_metar,is_wind_variable,weather_age_minutes
0,2026-04-17 22:01:45,2026-04-17,Arrival,21:30,21:29,Cần Thơ (VCA),VJ704,Đã đến,DN,CAN THO (VCA),other,2026-04-17 21:30:00,2026-04-17 21:29:00,-1.0,0.0,DN|Arrival|CAN THO (VCA)|VJ704|2026-04-17 21:3...,VVDN,2026-04-17T15:00:00.000Z,2026-04-17 22:00:00,27.0,25.0,130.0,7.0,6.0,BKN@800ft,METAR VVDN 171500Z 13007KT 9999 BKN008 27/25 Q...,0.0,1.75
1,2026-04-17 22:01:45,2026-04-17,Departure,09:35,09:35,Hải Phòng (HPH),VN1670,3-16,DN,HAI PHONG (HPH),other,2026-04-17 09:35:00,2026-04-17 09:35:00,0.0,0.0,DN|Departure|HAI PHONG (HPH)|VN1670|2026-04-17...,VVDN,2026-04-17T15:00:00.000Z,2026-04-17 22:00:00,27.0,25.0,130.0,7.0,6.0,BKN@800ft,METAR VVDN 171500Z 13007KT 9999 BKN008 27/25 Q...,0.0,1.75
2,2026-04-17 22:01:45,2026-04-17,Departure,09:30,10:10,Hải Phòng (HPH),VJ720,27-34,DN,HAI PHONG (HPH),other,2026-04-17 09:30:00,2026-04-17 10:10:00,40.0,1.0,DN|Departure|HAI PHONG (HPH)|VJ720|2026-04-17 ...,VVDN,2026-04-17T15:00:00.000Z,2026-04-17 22:00:00,27.0,25.0,130.0,7.0,6.0,BKN@800ft,METAR VVDN 171500Z 13007KT 9999 BKN008 27/25 Q...,0.0,1.75


## Bước 7 - Tạo feature cho model
Bước này tạo các feature thời gian, airline_code, route, và một số feature weather cơ bản. Sau đó lọc ra tập train có nhãn đầy đủ.

In [8]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()


    out["scheduled_hour"] = out["scheduled_dt"].dt.hour
    out["scheduled_dayofweek"] = out["scheduled_dt"].dt.dayofweek
    out["scheduled_month"] = out["scheduled_dt"].dt.month


    # Cyclical encoding (giống notebook training)
    out["sin_hour"] = np.sin(2 * np.pi * out["scheduled_hour"] / 24)
    out["cos_hour"] = np.cos(2 * np.pi * out["scheduled_hour"] / 24)


    out["airline_code"] = out["flight_number"].str.extract(r"^([A-Z0-9]{2})", expand=False)
    out["flight_num_only"] = pd.to_numeric(out["flight_number"].str.extract(r"(\d+)", expand=False), errors="coerce")


    out["minutes_to_departure_at_snapshot"] = (out["scheduled_dt"] - out["retrieved_at_vn"]).dt.total_seconds() / 60.0
    out["is_estimated_missing"] = out["estimated_dt"].isna().astype(int)


    # Defensive casts: một số cột weather có thể là object/str
    out["temperature_c"] = pd.to_numeric(out["temperature_c"], errors="coerce")
    out["dew_point_c"] = pd.to_numeric(out["dew_point_c"], errors="coerce")
    out["wind_speed_kt"] = pd.to_numeric(out["wind_speed_kt"], errors="coerce")
    out["visibility_miles"] = pd.to_numeric(out["visibility_miles"], errors="coerce")


    # Visibility binning (giống notebook training)
    out["visibility_bin"] = pd.cut(
        out["visibility_miles"],
        bins=[-np.inf, 3, 6, np.inf],
        labels=["Bad", "Medium", "Good"],
    ).astype(str)


    out["temp_dew_spread"] = out["temperature_c"] - out["dew_point_c"]
    out["is_low_visibility"] = (out["visibility_miles"] <= 3).astype(float)
    out["is_fog_risk"] = ((out["temp_dew_spread"] < 2) & (out["wind_speed_kt"] < 5)).astype(float)


    return out


def add_operational_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()


    def _congestion(g: pd.DataFrame) -> pd.DataFrame:
        g = g.sort_values("scheduled_dt")
        if g["scheduled_dt"].notna().any():
            s = g.set_index("scheduled_dt")
            g["airport_congestion_2h"] = s["flight_key"].rolling("2h", center=True).count().values
        else:
            g["airport_congestion_2h"] = np.nan
        return g


    def _rolling_delay(g: pd.DataFrame) -> pd.DataFrame:
        g = g.sort_values("retrieved_at_vn")
        if g["retrieved_at_vn"].notna().any():
            s = g.set_index("retrieved_at_vn")
            g["rolling_delay_rate_2h"] = s["label_delay"].rolling("2h", closed="left").mean().values
        else:
            g["rolling_delay_rate_2h"] = np.nan
        return g


    out = out.groupby("source_airport", group_keys=False).apply(_congestion)
    out = out.groupby("source_airport", group_keys=False).apply(_rolling_delay)
    return out


def add_target_encodings(train_df: pd.DataFrame, apply_df: pd.DataFrame) -> pd.DataFrame:
    out = apply_df.copy()
    global_rate = train_df["label_delay"].mean()
    route_rate = train_df.groupby("route_airport_std")["label_delay"].mean()
    airline_rate = train_df.groupby("airline_code")["label_delay"].mean()


    out["route_delay_rate"] = out["route_airport_std"].map(route_rate).fillna(global_rate)
    out["airline_historical_delay_rate"] = out["airline_code"].map(airline_rate).fillna(global_rate)
    return out


current_features = add_features(current_with_weather)
training_features = add_features(train_with_weather)


current_features = add_operational_features(current_features)
training_features = add_operational_features(training_features)


train_base = training_features[training_features["label_delay"].isin([0, 1])].copy()
current_features = add_target_encodings(train_base, current_features)
training_features = add_target_encodings(train_base, training_features)


# Keep only labeled rows for training
training_dataset = training_features[training_features["label_delay"].isin([0, 1])].copy()
training_dataset["label_delay"] = training_dataset["label_delay"].astype(int)


# # Fix data leakage (giống notebook training): chỉ dùng snapshot trước giờ bay
# n_before = len(training_dataset)
# training_dataset = training_dataset[training_dataset["minutes_to_departure_at_snapshot"] >= 0].copy()
# print("Dropped leakage rows (minutes_to_departure_at_snapshot < 0):", n_before - len(training_dataset))


print("Training dataset shape:", training_dataset.shape)
training_dataset[["source_airport", "flight_number", "scheduled_dt", "delay_minutes", "label_delay", "minutes_to_departure_at_snapshot", "visibility_miles", "visibility_bin"]].head(10)

Training dataset shape: (13042, 45)


,source_airport,flight_number,scheduled_dt,delay_minutes,label_delay,minutes_to_departure_at_snapshot,visibility_miles,visibility_bin
54,DN,VJ621,2026-04-17 05:35:00,0.0,0,-986.75,6.0,Medium
113,DN,VN140,2026-04-17 18:05:00,2.0,0,-236.75,6.0,Medium
39,DN,VN1941,2026-04-17 18:00:00,0.0,0,-241.75,6.0,Medium
44,DN,VN137,2026-04-17 17:55:00,0.0,0,-246.75,6.0,Medium
76,DN,VN179,2026-04-17 17:45:00,26.0,1,-256.75,6.0,Medium
77,DN,VN177,2026-04-17 17:35:00,9.0,0,-266.75,6.0,Medium
34,DN,9G983,2026-04-17 17:35:00,0.0,0,-266.75,6.0,Medium
88,DN,VN1440,2026-04-17 17:25:00,-4.0,0,-276.75,6.0,Medium
45,DN,VN135,2026-04-17 17:25:00,10.0,0,-276.75,6.0,Medium
111,DN,VN136,2026-04-17 17:20:00,5.0,0,-281.75,6.0,Medium


## Bước 8 - Lưu dữ liệu train vào PostgreSQL

Bước này chỉ ghi tập dữ liệu dùng để train model (`training_dataset_labeled`) lên PostgreSQL.

In [9]:
# Chuẩn hóa training_dataset trước khi lưu DB (đảm bảo có flight_key và schema phù hợp)
training_dataset_to_save = training_dataset.copy()

if "flight_key" not in training_dataset_to_save.columns:
    key_cols = ["source_airport", "direction", "route_airport_std", "flight_number", "scheduled_dt"]
    missing = [c for c in key_cols if c not in training_dataset_to_save.columns]
    if missing:
        raise KeyError(f"Khong the tao flight_key, thieu cot: {missing}")
    training_dataset_to_save["flight_key"] = training_dataset_to_save[key_cols].astype(str).agg("|".join, axis=1)


def ensure_table_has_columns(table_name: str, df: pd.DataFrame) -> None:
    """If table exists, add any missing columns as TEXT."""
    import psycopg2
    from psycopg2 import sql
    database_url = os.getenv("DATABASE_URL")
    if not database_url:
        raise RuntimeError("Missing DATABASE_URL environment variable")

    with psycopg2.connect(database_url) as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                SELECT EXISTS (
                    SELECT 1
                    FROM information_schema.tables
                    WHERE table_schema = 'public' AND table_name = %s
                );
                """,
                (table_name,),
            )
            table_exists = cur.fetchone()[0]
            if not table_exists:
                return

            cur.execute(
                """
                SELECT column_name
                FROM information_schema.columns
                WHERE table_schema = 'public' AND table_name = %s;
                """,
                (table_name,),
            )
            existing_cols = {row[0] for row in cur.fetchall()}

            for col in df.columns:
                if col in existing_cols:
                    continue
                alter = sql.SQL("ALTER TABLE {} ADD COLUMN {} TEXT").format(
                    sql.Identifier(table_name),
                    sql.Identifier(col),
                )
                cur.execute(alter)


outputs = {
    "training_dataset_labeled": (training_dataset_to_save, ["flight_key"]),
}

for table_name, (df_out, unique_cols) in outputs.items():
    ensure_table_has_columns(table_name, df_out)
    inserted = save_dataframe(df_out, table_name=table_name, unique_cols=unique_cols)
    print(f"Saved table: {table_name} | inserted={inserted} | shape={df_out.shape}")

Saved table: training_dataset_labeled | inserted=42 | shape=(13042, 45)


In [10]:
out_df = processed_dir / "training_dataset_labeled.csv"
training_dataset_to_save.to_csv(out_df)
print(f"saved {out_df}") 
training_dataset_to_save.shape

saved /mnt/c/Users/ADMIN/Desktop/IDA/data-monitoring-lab/Data Collection + Processing/data/processed/training_dataset_labeled.csv


(13042, 45)

## Bước 9 - Kiểm tra nhanh kết quả
Bước này thống kê nhanh tỷ lệ delay và số mẫu theo sân bay để xác nhận dữ liệu train có hợp lý.

In [11]:
summary_by_airport = training_dataset.groupby("source_airport").agg(
    samples=("label_delay", "count"),
    delay_rate=("label_delay", "mean"),
    avg_delay_minutes=("delay_minutes", "mean"),
)

summary_by_airport["delay_rate"] = (summary_by_airport["delay_rate"] * 100).round(2)
summary_by_airport["avg_delay_minutes"] = summary_by_airport["avg_delay_minutes"].round(2)

print("Tong so mau train:", len(training_dataset))
summary_by_airport

Tong so mau train: 13042


,samples,delay_rate,avg_delay_minutes
source_airport,,,
DN,3729,1.64,1.03
NB,7092,19.87,6.55
TSN,2221,3.42,-4.18


In [12]:
# --- QC report (missingness / weather match / lead-time sanity) ---
try:
    from IPython.display import display  # type: ignore
except Exception:
    def display(x):  # fallback ngoài notebook
        print(x)


def _qc_missing(df: pd.DataFrame, name: str, cols: list) -> pd.DataFrame:
    out = (
        df[cols]
        .isna()
        .mean()
        .mul(100)
        .round(2)
        .rename("missing_%")
        .to_frame()
        .sort_values("missing_%", ascending=False)
    )
    out.insert(0, "dataset", name)
    return out


qc_cols_flight = [
    "source_airport",
    "direction",
    "route_airport_std",
    "flight_number",
    "scheduled_dt",
    "retrieved_at_vn",
    "estimated_dt",
    "delay_minutes",
    "label_delay",
 ]
qc_cols_weather = [
    "report_time_vn",
    "temperature_c",
    "dew_point_c",
    "wind_direction_deg",
    "wind_speed_kt",
    "visibility_miles",
    "cloud_cover",
    "weather_age_minutes",
 ]

frames = []
for n, df_ in [
    ("flights_current", current_with_weather),
    ("flights_training_snapshot", train_with_weather),
    ("training_dataset", training_dataset),
 ]:
    cols_here = [c for c in (qc_cols_flight + qc_cols_weather) if c in df_.columns]
    frames.append(_qc_missing(df_, n, cols_here))

qc_missing = pd.concat(frames, axis=0).reset_index(names="column")
print("=== Missingness (%) by dataset ===")
display(qc_missing)

if "weather_age_minutes" in train_with_weather.columns:
    print("=== Weather age (minutes) on matched rows (train) ===")
    wx_age = train_with_weather.loc[train_with_weather["report_time_vn"].notna(), "weather_age_minutes"]
    display(wx_age.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame().T)

if "retrieved_at_vn" in training_dataset.columns and "scheduled_dt" in training_dataset.columns:
    print("=== Lead-time (minutes_to_departure_at_snapshot) on training_dataset ===")
    lead_min = (training_dataset["scheduled_dt"] - training_dataset["retrieved_at_vn"]).dt.total_seconds() / 60.0
    display(lead_min.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame().T)

print("=== Train label distribution ===")
display(training_dataset["label_delay"].value_counts(dropna=False).rename("count").to_frame())

=== Missingness (%) by dataset ===


,column,dataset,missing_%
0,delay_minutes,flights_current,33.69
1,estimated_dt,flights_current,33.66
2,label_delay,flights_current,32.69
3,temperature_c,flights_current,0.00
4,cloud_cover,flights_current,0.00
5,visibility_miles,flights_current,0.00
6,wind_speed_kt,flights_current,0.00
7,wind_direction_deg,flights_current,0.00
8,dew_point_c,flights_current,0.00
9,source_airport,flights_current,0.00


=== Weather age (minutes) on matched rows (train) ===


,count,mean,std,min,50%,90%,95%,99%,max
weather_age_minutes,19735.0,14.585965,8.844531,0.0,15.45,27.0,28.533333,29.566667,31.733333


=== Lead-time (minutes_to_departure_at_snapshot) on training_dataset ===


,count,mean,std,min,50%,90%,95%,99%,max
0,13042.0,256.880798,483.44884,-1311.633333,-9.116667,1036.5,1187.790833,1309.789,1434.233333


=== Train label distribution ===


,count
label_delay,
0,11496
1,1546
